# 🧠 Manasitra Custom Transformer — Google Colab High-Performance GPU Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashpatadiya1724-cmyk/mansitra-ai-or/blob/main/manasitra/ai-engine/custom_model/manasitra_train_colab.ipynb)

Train **Manasitra's Custom Emotional AI Transformer** on free Google Colab GPUs (NVIDIA T4 / V100 / A100).

### Supported Architecture Profiles:
- 📱 **`small`** (32M Parameters) — Ultra-fast mobile/edge inference (~5 min on T4)
- ⚖️ **`base`** (146M Parameters) — Default balanced psychology model (~15-20 min on T4)
- 🚀 **`large`** (372M Parameters) — High-capacity deep emotional reasoning (~40 min on T4)
- 👑 **`xlarge`** (758M Parameters) — State-of-the-art heavy foundation model (~1.5 hours on T4)

> **Prerequisite:** Make sure GPU is enabled in Colab menu: `Runtime` -> `Change runtime type` -> `T4 GPU`.

## 1. Verify NVIDIA GPU & CUDA Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total:      {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected! Please enable GPU in Runtime -> Change runtime type -> T4 GPU")

## 2. Clone Manasitra Repository & Set Up Working Directory

In [ ]:
import os, sys

# Clone or pull latest repository code
if not os.path.exists("mansitra-ai-or"):
    !git clone https://github.com/yashpatadiya1724-cmyk/mansitra-ai-or.git
%cd mansitra-ai-or/manasitra/ai-engine/custom_model

print(f"Current working directory: {os.getcwd()}")
!ls -la

## 3. Configure Training Hyperparameters

In [ ]:
#@title ⚙️ Training Configuration Form
MODEL_PROFILE = "large" #@param ["small", "base", "large", "xlarge"]
EPOCHS = 100 #@param {type:"slider", min:10, max:300, step:10}
BATCH_SIZE = 32 #@param {type:"slider", min:8, max:128, step:8}
LEARNING_RATE = 1.5e-4 #@param {type:"number"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}

from model import MODEL_PROFILES, get_model_profile
cfg = get_model_profile(MODEL_PROFILE)
print(f"Selected Architecture: Manasitra-{MODEL_PROFILE.upper()} ({cfg['parameters']} parameters)")
print(f"Config: d_model={cfg['d_model']}, nhead={cfg['nhead']}, enc_layers={cfg['enc_layers']}, dec_layers={cfg['dec_layers']}")
print(f"Epochs: {EPOCHS}, Batch Size: {BATCH_SIZE}, LR: {LEARNING_RATE}")

## 4. Run High-Speed CUDA GPU Training (with FP16 Mixed Precision)

In [ ]:
import os
os.environ["MODEL_PROFILE"] = MODEL_PROFILE
os.environ["EPOCHS"] = str(EPOCHS)
os.environ["BATCH_SIZE"] = str(BATCH_SIZE)
os.environ["LR"] = str(LEARNING_RATE)

# Execute optimized GPU training script
import train
train.train()
print("\n🎉 Training process completed successfully!")

## 5. Live Interactive Inference Test (Test Your Model Right Now!)

In [ ]:
from inference import generate

test_cases = [
    ("I am feeling so anxious about exams tomorrow", "anxiety"),
    ("Nobody talks to me in hostel and I feel lonely", "lonely"),
    ("College pressure is getting too heavy to handle", "overwhelmed"),
    ("I am not happy with my progress and feel empty", "sad"),
    ("I got placed today, feeling so grateful and happy", "happy"),
]

print("=" * 65)
print(f"  Live Inference Test with Manasitra-{MODEL_PROFILE.upper()}")
print("=" * 65)

for prompt, emotion in test_cases:
    reply = generate(prompt, emotion)
    print(f"\nUser [{emotion.upper()}]: {prompt}")
    print(f"Manasitra: {reply}")

## 6. Export Trained Model Artifacts to Google Drive or Local Download

In [ ]:
import shutil
from google.colab import files

# Create zip archive of trained weights, metadata, and tokenizer
output_zip = f"manasitra_{MODEL_PROFILE}_model"
shutil.make_archive(output_zip, 'zip', 'saved_model')
zip_filename = f"{output_zip}.zip"
print(f"Created archive: {zip_filename} ({os.path.getsize(zip_filename) / (1024*1024):.2f} MB)")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest_dir = "/content/drive/MyDrive/Manasitra_Model"
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy(zip_filename, os.path.join(dest_dir, zip_filename))
    print(f"✅ Successfully backed up to Google Drive at: {dest_dir}/{zip_filename}")
else:
    print("Downloading model zip to your local computer...")
    files.download(zip_filename)